In [17]:
# Load necessary libraries
suppressMessages(library(edgeR))
suppressMessages(library(DESeq2))
suppressMessages(library(ggplot2))
suppressMessages(library(latex2exp))
suppressMessages(library(pROC))
suppressMessages(library(data.table))
suppressMessages(library(glmnet))
suppressMessages(library(corrplot))
suppressMessages(library(caret))
suppressMessages(library(dplyr))

# Infering predictions

In [18]:
inference_design <- read.csv("/home/bdkhoi/projects/transcriptomics_classifier/data/test_data/design.csv", row.names=1)
inference_humancount <- read.csv("/home/bdkhoi/projects/transcriptomics_classifier/data/test_data/merged_counts_human/merged_gene_counts.csv", row.names=1)
inference_viruscount <- read.csv("/home/bdkhoi/projects/transcriptomics_classifier/data/test_data/merged_counts_virus/merged_gene_counts.csv", row.names=1)

In [19]:
inference_humancount <- inference_humancount[, rownames(inference_design), drop = FALSE]
inference_viruscount <- inference_viruscount[, rownames(inference_design), drop = FALSE]

In [20]:
coef_matrix <- read.csv("/home/bdkhoi/projects/transcriptomics_classifier/result/combined/coef_matrix.csv", row.names = 1)
coef_matrix <- coef_matrix[coef_matrix["s0"] != 0, , drop=FALSE]

In [21]:
inference_count <- rbind(inference_humancount, inference_viruscount)

In [22]:
selected_features <- rownames(coef_matrix)[coef_matrix[, 1] != 0]
selected_genes <- selected_features[!(selected_features == "(Intercept)")]
intercept <- coef_matrix["(Intercept)", ]
beta_gene_values <- coef_matrix[selected_genes, ]

In [23]:
missing_genes <- setdiff(selected_genes, rownames(inference_count))
print(missing_genes)

character(0)


In [24]:
rsrs <- inference_count[selected_genes, ]

In [25]:
coef_matrix

,s0
,<dbl>
(Intercept),1.797978201
ENSG00000233725.7,-0.049714616
ENSG00000134873.9,-0.140837256
ENSG00000171564.11,-0.075748870
ENSG00000237999.2,0.237969839
ENSG00000242948.1,-0.071465202
ENSG00000228519.3,0.080068432
ENSG00000213539.4,0.186228528
ENSG00000271417.2,-0.103219638


In [26]:
rsrs <- log2(rsrs + 1)

In [27]:
rsrs_score <- t(as.matrix(beta_gene_values)) %*% as.matrix(rsrs)
test_pred_score <- intercept + rsrs_score

In [28]:
mu <- exp(test_pred_score) / (1 + exp(test_pred_score))
y_test <- factor(inference_design$group, levels = c("normal", "cancer"))
y_test <- as.numeric(y_test) - 1
true_labels <- t(as.matrix(y_test))

In [29]:
mu

N_1,N_2,N_3,N_4,N_5,N_6,N_7,C_1,C_2,C_3,C_4,C_5,C_6,C_7,HPV_1,HPV_2,HPV_3
0.05345529,0.09937349,0.08157515,0.0301374,0.7542358,0.68505,0.6047729,0.2176284,0.4332355,0.5150715,0.4987078,0.7559681,0.683895,0.6488125,0.8746598,0.8258511,0.729397


In [30]:
roc_obj <- roc(true_labels, mu)
best_cutoff <- coords(roc_obj, "best", ret = "threshold")

Setting levels: control = 0, case = 1

Warning message in roc.default(true_labels, mu):
“Deprecated use a matrix as predictor. Unexpected results may be produced, please pass a numeric vector.”
Warning message in roc.default(true_labels, mu):
“Deprecated use a matrix as response. Unexpected results may be produced, please pass a vector or factor.”
Setting direction: controls < cases



In [31]:
as.factor(true_labels)

[1] 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1
Levels: 0 1

In [32]:
pred_labels <- ifelse(as.vector(mu) < as.numeric(best_cutoff[1]), 0, 1)
conf_matrix <- confusionMatrix(as.factor(pred_labels), as.factor(true_labels))
print(conf_matrix$overall["Accuracy"])
auc_value <- auc(roc_obj)
print(paste("AUC:", round(auc_value, 4)))

 Accuracy 
0.8235294 
[1] "AUC: 0.7571"


# Read RDS

In [8]:
load("data/112_samples/deseq2.dds.RData")

In [9]:
rds <- readRDS("data/112_samples/deseq2.rds")

In [10]:
count <- assays(rds)$counts

# Remove BE from PCA

In [ ]:
design <- read.csv("data/112_samples/design_full.csv", row.names=2)
head(design)

In [25]:
write.csv(design %>% mutate(cohort = ifelse(study == "GSE63514", "test", "train")), "data/112_samples/design_full.csv", row.names = TRUE)

In [11]:
count <- count[, rownames(design), drop=FALSE]

In [ ]:
dds <- DESeqDataSetFromMatrix(countData=count, colData=design, design=~group)

In [ ]:
vsd <- vst(dds)
plotPCA(vsd, "study")

In [ ]:
assay(vsd) <- limma::removeBatchEffect(assay(vsd), vsd$study)
plotPCA(vsd, "study")

In [ ]:
plotPCA(vsd, "group")

# Remove biologically related samples

In [1]:
sim_mat <- read.csv("data/112_samples/no_replicates/DESeq2_sample_similarity_matrix_pearson_111.tsv", sep="\t")
threshold <- 0.85

In [3]:
sample_list <- findCorrelation(sim_mat, cutoff = threshold)

In [ ]:
sample_keep <- colnames(sim_mat)[!(colnames(sim_mat) %in% colnames(sim_mat[sample_list]))]
length(sample_keep)

In [ ]:
viral_count <- read.csv("data/112_samples/clean_viral_counts.csv", row.names=1)
head(viral_count)

In [10]:
write.csv(viral_count[, (sample_keep), drop=FALSE], "data/112_samples/no_replicates/raw_viral_norep.csv")

In [12]:
write.csv(design[(sample_keep), ,drop=FALSE], "data/112_samples/no_replicates/design_norep.csv")

In [13]:
write.csv(count[, (sample_keep), drop=FALSE], "data/112_samples/no_replicates/raw_human_norep.csv")

# RDSPCA

In [ ]:
load("data/112_samples/deseq2.dds.RData")
ls()

In [18]:
rds <- readRDS("data/112_samples/deseq2.rds")
count <- assays(rds)$counts

In [ ]:
design <- read.csv("data/112_samples/design_112.csv")
head(design)

In [60]:
coldata <- data.frame(row.names=design$sample, group=design$group, study=design$study)

In [ ]:
countdata <- count[, rownames(coldata)]
head(countdata)

In [ ]:
dds <- DESeqDataSetFromMatrix(countData=countdata, colData=coldata, design=~group)

In [63]:
dds <- dds[rowSums(counts(dds)) > 0, ]
dds <- estimateSizeFactors(dds)
normalized_counts <- counts(dds,normalized=TRUE)

In [64]:
transformed <- vst(dds, blind=FALSE)

In [65]:
plotPCA_vst <- function (object,  ntop = 1000, assay=length(assays(object))) {
    rv         <- rowVars(assay(object, assay))
    select     <- order(rv, decreasing = TRUE)[seq_len(min(ntop, length(rv)))]
    pca        <- prcomp(t(assay(object, assay)[select, ]), center=TRUE, scale=FALSE)
    percentVar <- pca$sdev^2/sum(pca$sdev^2)
    df         <- cbind( as.data.frame(colData(object)), pca$x)
    ord        <- order(abs(rank(df$PC1)-median(df$PC1)), abs(rank(df$PC2)-median(df$PC2)))
    df         <- df[ord,]
    attr(df, "percentVar") <- data.frame(PC=seq(along=percentVar), percentVar=100*percentVar)
    return(df)
}

In [ ]:
vst_name <- "vst"
rld      <- varianceStabilizingTransformation(dds)
assay(dds, vst_name) <- assay(rld)
vst_name <- "vst"
ntop <- c(1000, Inf)
for (n_top_var in ntop) {
    pca.data      <- plotPCA_vst(dds, assay=vst_name, ntop=n_top_var)
    percentVar    <- round(attr(pca.data, "percentVar")$percentVar)
    plot_subtitle <- ifelse(n_top_var==Inf, "All genes", paste("Top", n_top_var, "genes"))
    pl <- ggplot(pca.data, aes(PC1, PC2, label=paste0(" ", colData(transformed)$study, " "))) +
        geom_point() +
        geom_text(check_overlap=TRUE, vjust=0.5, hjust="inward") +
        xlab(paste0("PC1: ",percentVar[1],"% variance")) +
        ylab(paste0("PC2: ",percentVar[2],"% variance")) +
        labs(title = paste0("First PCs on ", vst_name, "-transformed data"), subtitle = plot_subtitle) +
        theme(legend.position="top",
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            panel.background = element_blank(),
            panel.border = element_rect(colour = "black", fill=NA, size=1))
    print(pl)
}

In [ ]:
vst_name <- "vst"
rld      <- varianceStabilizingTransformation(dds)
assay(dds, vst_name) <- assay(rld)
vst_name <- "vst"
ntop <- c(1000, Inf)
for (n_top_var in ntop) {
    pca.data      <- plotPCA_vst(dds, assay=vst_name, ntop=n_top_var)
    percentVar    <- round(attr(pca.data, "percentVar")$percentVar)
    plot_subtitle <- ifelse(n_top_var==Inf, "All genes", paste("Top", n_top_var, "genes"))
    pl <- ggplot(pca.data, aes(PC1, PC2, label=paste0(" ", colnames(transformed), " "))) +
        geom_point() +
        geom_text(check_overlap=TRUE, vjust=0.5, hjust="inward") +
        xlab(paste0("PC1: ",percentVar[1],"% variance")) +
        ylab(paste0("PC2: ",percentVar[2],"% variance")) +
        labs(title = paste0("First PCs on ", vst_name, "-transformed data"), subtitle = plot_subtitle) +
        theme(legend.position="top",
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            panel.background = element_blank(),
            panel.border = element_rect(colour = "black", fill=NA, size=1))
    print(pl)
}